In [3]:
!git clone https://github.com/TencentAILabHealthcare/DNAGPT.git
%cd DNAGPT
!pip install -r requirements.txt

Cloning into 'DNAGPT'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 257 (delta 75), reused 211 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 228.43 KiB | 2.06 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/kaggle/working/DNAGPT
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 33.3 MB/s eta 0:00:0000:0100:01


In [4]:
%cd /kaggle/working/DNAGPT

/kaggle/working/DNAGPT


In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from time import time
import json
import torch

from dna_gpt.model import DNAGPT
from dna_gpt.tokenizer import KmerTokenizer
from dna_gpt.utils import seed_all_rng

def set_seed(seed: int = 42):
    np.random.seed(seed)
    seed_all_rng(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



class DNAGPTEmbeddingExtractor:
    """
    Извлечение эмбеддингов из DNAGPT.
    """
    @staticmethod
    def get_model_and_tokenizer(model_name: str):
        """
        Инициализация DNAGPT модели и KmerTokenizer.
        """
        special_tokens = (
            [str(i) for i in range(10)] + ['+', '-', '*', '/', '=', '&', '|', '!'] +
            ['M', 'B', 'P', 'R', 'I', 'K', 'L', 'O', 'Q', 'S', 'U', 'V', 'W', 'Y', 'X', 'Z']
        )
        dynamic = False if model_name == 'dna_gpt0.1b_h' else True
        tokenizer = KmerTokenizer(6, special_tokens, dynamic)
        model = DNAGPT.from_name(model_name, vocab_size=len(tokenizer))
        return model, tokenizer

    @staticmethod
    def load_weights(model, tokenizer, weight_path: str, device, dtype):
        state = torch.load(weight_path, map_location='cpu')
        sd = state.get('model', state)
        model.load_state_dict(sd, strict=False)
        model.to(device=device, dtype=dtype).eval()
        return model, tokenizer

    def __init__(self, model_name: str, weight_path: str, device=None, dtype=None, seed: int = 42):
        set_seed(seed)
        self.device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        self.dtype = dtype or (torch.float16 if self.device.type=='cuda' else torch.float32)
        model, tokenizer = get_model_and_tokenizer(model_name)
        self.model, self.tokenizer = load_weights(model, tokenizer, weight_path, self.device, self.dtype)
        self.max_len = getattr(self.model, 'max_len', None)

    def extract_embeddings(self, seqs, batch_size: int = 8):
        all_emb = []
        with torch.no_grad():
            for i in tqdm(range(0, len(seqs), batch_size), desc="Extracting embeddings"):
                batch = seqs[i:i+batch_size]
                encoded = [
                    self.tokenizer.encode(seq, max_len=self.max_len, device=self.device)
                    for seq in batch
                ]
                mlen = max(len(ids) for ids in encoded)
                input_ids = torch.full((len(encoded), mlen), self.tokenizer.pad_id, device=self.device, dtype=torch.long)
                attention_mask = torch.zeros_like(input_ids)
                for j, ids in enumerate(encoded):
                    input_ids[j, :len(ids)] = torch.tensor(ids, device=self.device)
                    attention_mask[j, :len(ids)] = 1

                outputs = self.model(
                    input_ids,
                    attention_mask
                )

                hidden = outputs[-1]
                mask = attention_mask.unsqueeze(-1)
                summed = (hidden * mask).sum(dim=1)
                lengths = mask.sum(dim=1).clamp(min=1)
                emb = (summed / lengths).cpu().numpy()
                all_emb.append(emb)
        return np.vstack(all_emb)


# Параметры DNAGPT
MODEL_NAME  = 'dna_gpt0.1b_h'
WEIGHT_PATH = ''
PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}

ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", trust_remote_code=True)
train_ds, test_ds = ds['train'], ds['test']

extractor = DNAGPTEmbeddingExtractor(MODEL_NAME, WEIGHT_PATH)

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task'] == t)
    te = test_ds.filter(lambda x, t=task: x['task'] == t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=16)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=16)

    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1,1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
    Xt = X_te.reshape(-1,1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'results_dnagpt_task-{task}_baseline.json', 'w') as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, ks=(1,5,10,20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task'] == t)
        te = test.filter(lambda x, t=task: x['task'] == t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])

        X_te = extractor.extract_embeddings(seqs_te, batch_size=16)
        res[task] = {}

        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())

                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=16)
                y_k = y_tr[idxs]

                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1,1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1,1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)

                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))

            res[task][k] = {
                'accuracy': float(np.mean(accs)),
                'f1_score': float(np.mean(f1s))
            }
            with open(f'results_dnagpt_task-{task}_k-{k}.json', 'w') as f:
                json.dump(res, f, indent=4)

    return res

results_kshot = few_shot(train_ds, test_ds)

output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open('results_dnagpt.json', 'w') as f:
    json.dump(output, f, indent=4)

print("Done! Results saved to results_dnagpt.json")


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

train.parquet:   0%|          | 0.00/8.13M [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/7.38M [00:00<?, ?B/s]

...

Filter:   0%|          | 0/48797 [00:00<?, ? examples/s]


Extracting embeddings:   0%|          | 0/138 [00:00<?, ?it/s]/tmp/ipykernel_35/3768476577.py:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids[j, :len(ids)] = torch.tensor(ids, device=self.device)

Extracting embeddings: 100%|██████████| 138/138 [00:07<00:00, 19.03it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 104.40it/s]

Extracting embeddings:   0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_35/3768476577.py:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids[j, :len(ids)] = torch.tensor(ids, device=self.device)
Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 98.96it/s]

Extracting embeddings:   0%|          | 0/1 [00:00<?, ?it/s]/tmp/

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48797 [00:00<?, ? examples/s]


Extracting embeddings:   0%|          | 0/230 [00:00<?, ?it/s]/tmp/ipykernel_35/3768476577.py:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids[j, :len(ids)] = torch.tensor(ids, device=self.device)

Extracting embeddings: 100%|██████████| 230/230 [00:10<00:00, 22.56it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 113.20it/s]